In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_csv_path = "/content/drive/MyDrive/TicketMind/data/train_data.csv"

In [ ]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0,
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
def get_sentiment(text: str):
    result = sentiment_analyzer(text)[0]
    return {
        "sentiment": result["label"].lower(),
        "confidence": round(result["score"], 4),
    }

In [ ]:
print(get_sentiment("I have a question about canceling my order"))
print(get_sentiment("This is absolutely terrible, I want a refund now"))
print(get_sentiment("Thanks, that solved my problem perfectly"))

{'sentiment': 'neutral', 'confidence': 0.5786}
{'sentiment': 'negative', 'confidence': 0.9475}
{'sentiment': 'positive', 'confidence': 0.9015}


In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

train_df = pd.read_csv(train_csv_path)

instructions_list = train_df["instruction"].tolist()
instruction_embeddings = embedding_model.encode(
    instructions_list,
    show_progress_bar=True,
    batch_size=64,
)

import numpy as np
save_path = "/content/drive/MyDrive/TicketMind/data/"
np.save(save_path + "instruction_embeddings.npy", instruction_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/308 [00:00<?, ?it/s]

In [ ]:
import numpy as np

save_path = "/content/drive/MyDrive/TicketMind/data/"
np.save(save_path + "instruction_embeddings.npy", instruction_embeddings)

print("embeddings saved")

embeddings saved


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_response(new_message: str, top_k: int = 3):
    new_embedding = embedding_model.encode([new_message])
    similarities = cosine_similarity(new_embedding, instruction_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "matched_instruction": train_df.iloc[idx]["instruction"],
            "suggested_response": train_df.iloc[idx]["response"],
            "similarity_score": round(float(similarities[idx]), 4),
        })
    return results

In [ ]:
results = find_similar_response("I want to cancel my order, how can I do that?", top_k=3)

for r in results:
    print(f"تشابه: {r['similarity_score']}")
    print(f"رسالة مشابهة: {r['matched_instruction']}")
    print(f"الرد المقترح: {r['suggested_response']}\n")

تشابه: 0.8726
رسالة مشابهة: I bought some product, I want to cancel order {{Order Number}}
الرد المقترح: I've come to understand that you've made a purchase and now you wish to cancel the order with the order number {{Order Number}}. Our cancellations team is here to help you with that. To proceed with the cancellation, you can follow these steps:

1. Access Your Account: Log in to your {{Online Company Portal Info}} using your credentials.
2. Locate Your Orders: Once you're logged in, navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Find the Specific Purchase: Look for the order number {{Order Number}} within the order history and click on it.
4. Initiate the Cancellation: You should see an option labeled '{{Online Order Interaction}}' associated with the purchase. Please select this option.
5. Complete the Process: The system may require additional information or confirmation to finalize the cancellation. Please provide the necessary detail

In [8]:
# 3) التنزيل المباشر (من غير الاعتماد على القائمة الجانبية)
from google.colab import files
files.download("/content/api_package.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>